# Problem 17

- [ ] Trace through `coprime_reconstruct_frequency` for inputs $M=7$, $N=11$, $k_n=5$, and $k_m=3$ and verify your hand calculations from parts a) and b).

Convention: Channel M samples every M steps and its FFT peak is k_n = k mod N; Channel N samples every N steps and its FFT peak is k_m = k mod M. A stride S observes k mod (MN/S).

For coherent fine-grid tones, f_k = k/(MN T_S), k=0,...,MN-1, where T_S is the base integration timestep. With M=7, N=11, and k=38, (k_m, k_n)=(3, 5), (a, b)=(-3, 2), and CRT reconstructs 38.

Color-code the CLI output with the public helpers and channel colors from `kuramoto.cli.style.domain`. Keep palette assignments centralized there rather than defining notebook-local CLI colors.

In [3]:
from __future__ import annotations

from kuramoto.cli.style import (
    PRESETS,
    cli_echo,
    cli_divider,
    cli_section,
    multi_gradient,
    style_coprime_coeff,
    style_coprime_gcd,
    style_coprime_remainder,
    style_coprime_term,
    style_crt_channel,
)
from kuramoto.solvers.coprime import coprime_reconstruct_frequency, extended_gcd

CHANNEL_M = 'M'
CHANNEL_N = 'N'

def trace_crt_reconstruction(
    k_m: int,
    k_n: int,
    M: int,
    N: int,
) -> int:
    '''Trace reconstruction from the crossed FFT aliases.

    Channel M yields k_n modulo N; Channel N yields k_m modulo M.
    '''
    modulus = M * N
    cli_section(f'CRT Reconstruction: ({style_coprime_term(M)}, {style_coprime_term(N)})')

    label_bezout = multi_gradient('Step 1 — Bézout Identity:', PRESETS['eigenscribe'])
    cli_echo(f'  {label_bezout}')
    g, a, b = extended_gcd(M, N)
    cli_echo(
        f'    ({style_coprime_coeff(a)}){style_coprime_term(M)} + ({style_coprime_coeff(b)}){style_coprime_term(N)} = {style_coprime_gcd(g)}'
    )
    label_a = style_coprime_coeff('a')
    label_b = style_coprime_coeff('b')
    cli_echo(
        f'    Bézout coefficients: {label_a}={style_coprime_coeff(a)}, '
        f'{label_b}={style_coprime_coeff(b)}'
    )
    cli_divider()

    label_n = multi_gradient('Step 2 — k_n contribution (Channel M FFT peak):', PRESETS['tse-gradient-2'])
    cli_echo(f'  {label_n}')
    term_n = k_n * a * M
    styled_term_n = style_crt_channel(term_n, CHANNEL_M)
    cli_echo(
        f'    Term k_n = {style_coprime_remainder(k_n)} × ({style_coprime_coeff(a)}) × {style_coprime_term(M)} '
        f'= {styled_term_n}'
    )
    cli_echo(
        f'    (Recovers remainder mod {style_coprime_term(N)}: '
        f'{style_coprime_coeff(a)} × {style_coprime_term(M)} ≡ 1 (mod {style_coprime_term(N)}))'
    )
    cli_divider()

    label_m = multi_gradient('Step 3 — k_m contribution (Channel N FFT peak):', PRESETS['tse-gradient-2'])
    cli_echo(f'  {label_m}')
    term_m = k_m * b * N
    styled_term_m = style_crt_channel(term_m, CHANNEL_N)
    cli_echo(
        f'    Term k_m = {style_coprime_remainder(k_m)} × ({style_coprime_coeff(b)}) × {style_coprime_term(N)} '
        f'= {styled_term_m}'
    )
    cli_echo(
        f'    (Recovers remainder mod {style_coprime_term(M)}: '
        f'{style_coprime_coeff(b)} × {style_coprime_term(N)} ≡ 1 (mod {style_coprime_term(M)}))'
    )
    cli_divider()

    label_merge = multi_gradient('Step 4 — Merge:', PRESETS['tse-gradient-2'])
    cli_echo(f'  {label_merge}')
    k_true = (term_m + term_n) % modulus
    cli_echo(
        f'    k_true = ({styled_term_m} + {styled_term_n}) % {style_coprime_term(modulus)} '
        f'= {style_coprime_gcd(k_true)}'
    )
    cli_divider()

    label_verify = multi_gradient('Step 5 — Verification:', PRESETS['tse-gradient-2'])
    cli_echo(f'  {label_verify}')
    check_m = k_true % M
    check_n = k_true % N
    match_m = '✔️' if check_m == k_m else '✖️'
    match_n = '✔️' if check_n == k_n else '✖️'
    cli_echo(
        f'    {style_coprime_gcd(k_true)} mod {style_coprime_term(M)} = {style_coprime_gcd(check_m)}  '
        f'{match_m} (Channel N FFT peak k_m={style_coprime_remainder(k_m)})'
    )
    cli_echo(
        f'    {style_coprime_gcd(k_true)} mod {style_coprime_term(N)} = {style_coprime_gcd(check_n)}  '
        f'{match_n} (Channel M FFT peak k_n={style_coprime_remainder(k_n)})'
    )
    cli_divider()

    k_lib = coprime_reconstruct_frequency(k_m=k_m, k_n=k_n, M=M, N=N)
    lib_match = '✔️' if k_lib == k_true else '✖️'
    cli_echo(
        f'  Library coprime_reconstruct_frequency() = {style_coprime_gcd(k_lib)}  {lib_match}'
    )
    return k_true

def trace_crt_reconstruction_error(
    k_m: int,
    k_n: int,
    M: int,
    N: int,
) -> None:
    '''Trace CRT ambiguity when M and N are not coprime.'''
    cli_section(f'CRT Reconstruction (Non-Coprime): ({style_coprime_term(M)}, {style_coprime_term(N)})')
    g, _, _ = extended_gcd(M, N)
    cli_echo(
        f'  gcd({style_coprime_term(M)}, {style_coprime_term(N)}) = {style_coprime_gcd(g)}'
    )
    if g != 1:
        cli_echo(
            f'  ✖️ Strides are not coprime (gcd = {style_coprime_gcd(g)}). '
            f'CRT reconstruction is ambiguous.'
        )
        cli_echo(
            f'  Multiple indices in [0, {M * N - 1}] share the same remainder pair '
            f'({style_coprime_remainder(k_m)}, {style_coprime_remainder(k_n)}).'
        )
        ambiguous = [t for t in range(M * N) if t % M == k_m and t % N == k_n]
        if ambiguous:
            cli_echo(f'  Matching indices: {ambiguous}')
        else:
            cli_echo('  No index matches — the remainder pair is inconsistent.')
    else:
        cli_echo('  ✔️ Strides are coprime. Proceeding with reconstruction.')
    cli_divider()

    try:
        k_true = coprime_reconstruct_frequency(k_m=k_m, k_n=k_n, M=M, N=N)
        cli_echo(f'  Result: {style_coprime_gcd(k_true)}')
    except ValueError as exc:
        cli_echo(f'  ValueError: {exc}')

In [4]:
# Problem 17: Channel M FFT peak k_n=5 (mod 11); Channel N FFT peak k_m=3 (mod 7)
k_true = trace_crt_reconstruction(k_m=3, k_n=5, M=7, N=11)
print(f'\nReconstructed k_true = {k_true}')


 ──────────────────────────────────────────────────
  CRT Reconstruction: (7, 11)
 ──────────────────────────────────────────────────
  Step 1 — Bézout Identity:
    (-3)7 + (2)11 = 1
    Bézout coefficients: a=-3, b=2
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 2 — k_n contribution (Channel M FFT peak):
    Term k_n = 5 × (-3) × 7 = -105
    (Recovers remainder mod 11: -3 × 7 ≡ 1 (mod 11))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 3 — k_m contribution (Channel N FFT peak):
    Term k_m = 3 × (2) × 11 = 66
    (Recovers remainder mod 7: 2 × 11 ≡ 1 (mod 7))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 4 — Merge:
    k_true = (66 + -105) % 77 = 38
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 5 — Verification:
    38 mod 7 = 3  ✔️ (Channel N FFT peak k_m=3)
    38 mod 11 = 5  ✔️ (Channel M FFT peak k_n=5)
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Library coprime_reconstruct_frequency() = 38  ✔️

Reconstructed k_true = 38


In [5]:
# Non-coprime counterexample
trace_crt_reconstruction_error(k_m=0, k_n=0, M=6, N=9)


 ──────────────────────────────────────────────────
  CRT Reconstruction (Non-Coprime): (6, 9)
 ──────────────────────────────────────────────────
  gcd(6, 9) = 3
  ✖️ Strides are not coprime (gcd = 3). CRT reconstruction is ambiguous.
  Multiple indices in [0, 53] share the same remainder pair (0, 0).
  Matching indices: [0, 18, 36]
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  ValueError: Strides M=6 and N=9 are not coprime (gcd = 3). CRT frequency reconstruction requires gcd(M, N) == 1.


In [6]:
# Another coprime example
k_true_2 = trace_crt_reconstruction(k_m=2, k_n=7, M=5, N=8)
print(f"\nReconstructed k_true = {k_true_2}")


 ──────────────────────────────────────────────────
  CRT Reconstruction: (5, 8)
 ──────────────────────────────────────────────────
  Step 1 — Bézout Identity:
    (-3)5 + (2)8 = 1
    Bézout coefficients: a=-3, b=2
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 2 — k_n contribution (Channel M FFT peak):
    Term k_n = 7 × (-3) × 5 = -105
    (Recovers remainder mod 8: -3 × 5 ≡ 1 (mod 8))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 3 — k_m contribution (Channel N FFT peak):
    Term k_m = 2 × (2) × 8 = 32
    (Recovers remainder mod 5: 2 × 8 ≡ 1 (mod 5))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 4 — Merge:
    k_true = (32 + -105) % 40 = 7
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 5 — Verification:
    7 mod 5 = 2  ✔️ (Channel N FFT peak k_m=2)
    7 mod 8 = 7  ✔️ (Channel M FFT peak k_n=7)
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Library coprime_reconstruct_frequency() = 7  ✔️

Reconstructed k_true = 7


## Part 2
- [ ] Print the channel stride samples and a color-coded CLI block chart for all 77 fine-grid bins. Channel M collapses bins into its 7-slot grid (`t mod 7`); Channel N collapses them into its 11-slot grid (`t mod 11`). Each chart column is a coarse slot, so repeated colored blocks make the collapsed classes and wrap-around visible.

In [7]:
from __future__ import annotations

import sys
from pathlib import Path

_here = Path.cwd()
_candidates = (
    _here,
    _here / 'SECTION_C' / 'CODING_TASKS',
    _here / 'notebooks' / 'practice-problems' / 'SECTION_C' / 'CODING_TASKS',
)
_mod_dir = next(
    candidate for candidate in _candidates
    if (candidate / 'crt_stride_visualization.py').is_file()
)
if str(_mod_dir) not in sys.path:
    sys.path.insert(0, str(_mod_dir))

from crt_stride_visualization import coprime_snapshot_steps
from kuramoto.cli.style import (
    cli_divider,
    cli_echo,
    cli_section,
    style_crt_channel,
)


def _build_header(m: int, n: int, total: int) -> str:
    channel_m = style_crt_channel(f'm={m}', 'M')
    channel_n = style_crt_channel(f'n={n}', 'N')
    return f'Coprime Snapshot Strides ({channel_m}, {channel_n}, period={total})'


def _format_channel_steps(steps: list[int], channel: str) -> str:
    values = (style_crt_channel(step, channel) for step in steps)
    return '[' + ', '.join(values) + ']'


def _print_channel_chart(
    channel: str,
    slots: int,
    wraps: int,
    total: int,
) -> None:
    cli_section(
        f'Channel {style_crt_channel(channel, channel)} — {slots}-slot coarse grid '
        f'(each slot receives {total // slots} fine bins)'
    )
    header = 'wrap ↓ | ' + ' '.join(
        style_crt_channel(f'[{slot:02d}]', channel)
        for slot in range(slots)
    )
    cli_echo(f'  {header}')

    for wrap in range(wraps):
        cells = (
            style_crt_channel(f'[{wrap * slots + slot:02d}]', channel)
            for slot in range(slots)
        )
        cli_echo(f'  {wrap:02d}    | ' + ' '.join(cells))

    cli_echo(
        f'  Columns are collapsed classes: fine bin t belongs to slot t mod {slots}.'
    )
    cli_divider()


def _print_wraparound_charts(m: int, n: int) -> None:
    total = m * n
    _print_channel_chart(channel='M', slots=m, wraps=n, total=total)
    _print_channel_chart(channel='N', slots=n, wraps=m, total=total)


def print_coprime_strides(m: int = 7, n: int = 11) -> None:
    chan_m, chan_n, union_steps = coprime_snapshot_steps(m, n)
    total = m * n

    cli_section(_build_header(m, n, total))
    label_m = f'Channel M (every {m}-th step) — {len(chan_m)} snapshots:'
    cli_echo('  ' + style_crt_channel(label_m, 'M'))
    cli_echo('    ' + _format_channel_steps(chan_m, 'M'))
    cli_divider()
    label_n = f'Channel N (every {n}-th step) — {len(chan_n)} snapshots:'
    cli_echo('  ' + style_crt_channel(label_n, 'N'))
    cli_echo('    ' + _format_channel_steps(chan_n, 'N'))
    cli_divider()
    overlap = sorted(set(chan_m) & set(chan_n))
    cli_echo(f'  Coprime union: {len(union_steps)} snapshots; overlap: {overlap}')
    cli_divider()

    _print_wraparound_charts(m, n)


print_coprime_strides(m=7, n=11)


 ──────────────────────────────────────────────────
  Coprime Snapshot Strides (m=7, n=11, period=77)
 ──────────────────────────────────────────────────
  Channel M (every 7-th step) — 12 snapshots:
    [0, 7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77]
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Channel N (every 11-th step) — 8 snapshots:
    [0, 11, 22, 33, 44, 55, 66, 77]
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Coprime union: 18 snapshots; overlap: [0, 77]
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─

 ──────────────────────────────────────────────────
  Channel M — 7-slot coarse grid (each slot receives 11 fine bins)
 ──────────────────────────────────────────────────
  wrap ↓ | [00] [01] [02] [03] [04] [05] [06]
  00    | [00] [01] [02] [03] [04] [05] [06]
  01    | [07] [08] [09] [10] [11] [12] [13]
  02    | [14] [15] [16] [17] [18] [19] [20]
  03    | [21] [22] [23] [24] [25] [26] [27]
  04    | [28] [29] [30] [31] [32] [33] [34]
  05    | [35] [36] [3

## Part 3: CRT coprime-stride diagnostics (CLI)

Verify the 7 × 11 CRT hit-map and snapshot bookkeeping in the terminal. The colored wrap-around grids in Part 2 replace the former matplotlib panels.

### 2.A Is my sampling correct? - CRT hit-map
- [ ] Verify that each cell in the 7 × 11 CRT grid maps to one distinct fine-grid bin in `[0, 77)`. Part 2 prints the complete color-coded mapping as two wrap-around block charts, so repeated coarse-slot classes are visible without matplotlib.

In [8]:
from __future__ import annotations

from kuramoto.cli.style import (
    cli_divider,
    cli_echo,
    cli_section,
    style_coprime_term,
    style_crt_channel,
)

M, N = 7, 11
TOTAL = M * N


def crt_cell_to_step(a: int, b: int) -> int:
    inv_n_mod_m = pow(N, -1, M)
    inv_m_mod_n = pow(M, -1, N)
    return (a * N * inv_n_mod_m + b * M * inv_m_mod_n) % TOTAL


def build_step_grid() -> list[list[int]]:
    return [[crt_cell_to_step(a, b) for b in range(N)] for a in range(M)]

In [9]:
def main() -> None:
    step_grid = build_step_grid()
    flat_steps = [step for row in step_grid for step in row]
    assert len(step_grid) == M
    assert all(len(row) == N for row in step_grid)
    assert sorted(flat_steps) == list(range(TOTAL))

    cli_section('CRT hit-map sanity check')
    cli_echo(
        f'CRT hit-map: {style_coprime_term(len(flat_steps))} residue cells cover '
        f'all {TOTAL} fine bins exactly once.'
    )

    dense_count = TOTAL + 1
    chan_m = list(range(0, TOTAL + 1, M))
    chan_n = list(range(0, TOTAL + 1, N))
    union_count = len(set(chan_m) | set(chan_n))
    saved = dense_count - union_count
    saved_percent = 100.0 * saved / dense_count

    cli_section('Snapshot bookkeeping')
    cli_echo(f'Dense snapshots: {dense_count}')
    label_m = style_crt_channel('Channel M', 'M')
    label_n = style_crt_channel('Channel N', 'N')
    cli_echo(f'{label_m} fires at t = 0 (mod {M}): {len(chan_m)} samples')
    cli_echo(f'{label_n} fires at t = 0 (mod {N}): {len(chan_n)} samples')
    cli_echo(
        f'Coprime union: {union_count} unique times '
        f'(overlap only at t = 0 (mod {TOTAL}))'
    )
    styled_percent = style_coprime_term(f'{saved_percent:.1f}%')
    cli_echo(f'Saved vs dense: {saved} timestamps ({styled_percent})')

    cli_section('Memory bookkeeping')
    cli_echo(
        'dense: O(mn) = 77 residue pairs | '
        'CRT: O(1) per queried residue via modular inverses'
    )
    cli_divider()


main()


 ──────────────────────────────────────────────────
  CRT hit-map sanity check
 ──────────────────────────────────────────────────
CRT hit-map: 77 residue cells cover all 77 fine bins exactly once.

 ──────────────────────────────────────────────────
  Snapshot bookkeeping
 ──────────────────────────────────────────────────
Dense snapshots: 78
Channel M fires at t = 0 (mod 7): 12 samples
Channel N fires at t = 0 (mod 11): 8 samples
Coprime union: 18 unique times (overlap only at t = 0 (mod 77))
Saved vs dense: 60 timestamps (76.9%)

 ──────────────────────────────────────────────────
  Memory bookkeeping
 ──────────────────────────────────────────────────
dense: O(mn) = 77 residue pairs | CRT: O(1) per queried residue via modular inverses
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
